In [ ]:
import numpy as np
from numpy.typing import NDArray

arr = np.array

In [ ]:
from matplotlib.cm import tab10
import ipywidgets as pw

import pyvista as pv

# pv.set_jupyter_backend("server")
tab10colors = tab10.colors  # type: ignore

# B Spline Curves


Everything is polinomial

Power vector $W^{(d)}(t) = \begin{bmatrix} 1 & t & \cdots & t^d \end{bmatrix}$


## Splines

For a span of knot vector with $t_i \le t \lt t_{i+1}$, the span supports $d+1$ splines: $B_{i-d},\dots,B_{i}$

$$
\begin{bmatrix} B^{(d)}_{i-d} & \cdots & B^{(d)}_{i} \end{bmatrix} =
 W^{(d)} M^{(d)}_i
$$

## Curves

With control points in $\mathbb{R}^2$ or $\mathbb{R}^3$ or anything for each spline

$
C^{(d)}_i = \begin{bmatrix}
c_{i-d} \\
\vdots \\
c_{i-1} \\
c_{i} \\
\end{bmatrix}
$

$$
S^{(d)}_i(t) =  W^{(d)} M^{(d)} C^{(d)}_i
$$

> the matrix multiplication with the $C$ is point-wise as if it's normal$


## B-Matrices

With uniform knot vector of $t_i = i$, and segment-localized $0 \le t \lt 1$ they are independant of the $i$ and the $t$

$$
M^{(1)} =
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix}
$$

$$
M^{(2)} =
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
$$

$$
M^{(3)} =
\frac{1}{6}
\begin{bmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{bmatrix}
$$


$M^{(1)} C^{(1)}_i =
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix}
\begin{bmatrix}
c_{i-1} \\
c_i
\end{bmatrix} =
\begin{bmatrix}
c_{i - 1} \\
c_{i} - c_{i - 1}
\end{bmatrix}
$

$M^{(2)} C^{(2)}_i = 
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
\begin{bmatrix}
c_{i-2} \\
c_{i-1} \\
c_i
\end{bmatrix} =
\frac{1}{2}
\begin{bmatrix}
c_{i-2} + c_{i-1} \\
-2 c_{i-2} + 2 c_{i-1} \\
c_{i-2} - 2c_{i-1} + c_{i}
\end{bmatrix}
$

$W^{(1)} M^{(1)} = 
\begin{bmatrix} 1 & t \end{bmatrix}
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix} = 
\big[ 1 - t, t \big]
$ (obviously)

$W^{(2)} M^{(2)} = 
\begin{bmatrix} 1 & t & t^2 \end{bmatrix}
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix} =
\frac{1}{2} \big[1 - 2 t + t^2, 1 + 2 t - 2 t^2, t^2 \big]
$


---


#### The $M^{(d)}$


In [ ]:
# blending matrices for uniform splines
BM = {
    1: np.array([[1, 0], [-1, 1]]),
    2: np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2,
    3: np.array([[1, 4, 1, 0], [-3, 0, 3, 0], [3, -6, 3, 0], [-1, 3, -3, 1]]) / 6,
}

#### The $W^{(d)}(t)$

$= \begin{bmatrix} 1 & t & t^2 & \cdots & t^d \end{bmatrix}$


In [ ]:
def powers_t(d: int, t: float):
    exps = np.arange(d + 1)
    return t**exps


def powers_tt(d: int, t: NDArray):
    exps = np.arange(d + 1)
    return t[..., None] ** exps[None, ...]

#### The $\begin{bmatrix} B_{i-d} & \cdots & B_{i} \end{bmatrix}$

$= W^{(d)} M^{(d)}$


In [ ]:
def splines_t(d: int, t: float):
    """[B_i-d ... B_i] = [0, 1, ... t^d] M"""
    return powers_t(d, t) @ BM[d]


def splines_tt(d: int, tt: NDArray):
    """[B_i-d ... B_i] = [0, 1, ... t^d] M"""
    return powers_tt(d, tt) @ BM[d]

#### The $C^{(d)}_i$

$= \begin{bmatrix}
c_{i-d} \\
c_{i-d+1} \\
\vdots \\
c_{i}
\end{bmatrix}$


In [ ]:
def controls_i(d: int, points: NDArray, i: int):
    """Slice of control points for i'th span of t
    i.e. for a spline starting at t_i
    """
    assert i >= d and i < len(points)
    return points[i - d : i + 1]

#### The $S^{(d)}_i$

$= W^{(d)} M^{(d)} C^{(d)}_i$


In [ ]:
def curve_t(d: int, controls: NDArray, t: float):
    """A point on curve at t, with 0 < t < 1"""
    assert controls.shape[0] == d + 1
    return powers_t(d, t) @ BM[d] @ controls


def curve_tt(d: int, controls: NDArray, tt: NDArray):
    """A segment of curve over tt, with 0 < t < 1"""
    assert controls.shape[0] == d + 1
    return powers_tt(d, tt) @ BM[d] @ controls

$W^{(d)} M^{(d)} \big[ C^{(d)}_i |_d^n \big]$

Through all the points


In [ ]:
def megacurve(d: int, points: NDArray, tt: NDArray):
    wm = powers_tt(d, tt) @ BM[d]
    return np.concat([wm @ controls_i(d, points, i) for i in range(d, len(points))])

---


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
def mycurve(points):
    return megacurve(3, points, np.arange(0, 1.0, 0.125))

In [ ]:
points = arr([
    np.arange(0, 10),
    np.random.random(10) * 5,
    np.random.random(10) * 5,
]).T

In [ ]:
pvm_controls = pv.lines_from_points(points)
pvm_spline = pv.lines_from_points(mycurve(points))

In [ ]:
pl = pv.Plotter()
pl.background_color = pv.Color("#303030")
pl.show_grid()
pva_controls = pl.add_mesh(pvm_controls, color="gray", line_width=2)
pva_spline = pl.add_mesh(pvm_spline, color="white", line_width=4)
pl.view_xy()
pl.show()

In [ ]:
def update(point, idx):
    points[idx] = point
    pvm_controls.points = points
    pvm_spline.points = mycurve(points)

In [ ]:
_ = pl.add_sphere_widget(update, center=points, radius=0.125)

---


## Derivatives

- $T = \frac{dS}{dt}$ — tangent in direction of curve.

$\frac{dW^{(d)}}{dt} = \begin{bmatrix} 0 & 1 & & 2t & \cdots & (d)t^{d-1} \end{bmatrix}$

$(S^{(1)}_i)' =  
\begin{bmatrix} 0 & 1 \end{bmatrix}
\begin{bmatrix} 1 & 0 \\ -1 & 1\end{bmatrix}
C_i =
\begin{bmatrix} -1 & 1\end{bmatrix}
C_i 
$

$(S^{(2)}_i)' =  
\begin{bmatrix} 0 & 1 & 2t \end{bmatrix}
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
C_i =
\begin{bmatrix} 1 & t \end{bmatrix}
\begin{bmatrix}
-1 & 1 & 0 \\
1 & -2 & 1
\end{bmatrix}
C_i
$

$(S^{(3)}_i)' =  
\begin{bmatrix} 0 & 1 & 2t & 3t^2 \end{bmatrix}
\frac{1}{6}
\begin{bmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{bmatrix}
C_i =
\begin{bmatrix} 1 & t & t^2 \end{bmatrix}
\frac{1}{2}
\begin{bmatrix}
-1 & 0 & 1 & 0 \\
2 & -4 & 2 & 0 \\
-1 & 3 & -3 & 1
\end{bmatrix}
C_i$


In [ ]:
BMdt = {
    1: np.array([[-1, 1]]),
    2: np.array([[-1, 1, 0], [1, -2, 1]]),
    3: np.array([[-1, 0, 1, 0], [2, -4, 2, 0], [-1, 3, -3, 1]]) / 2,
}

In [ ]:
def dir(d: int, controls: NDArray, t: float):
    """A tangent on curve at t, with 0 < t < 1"""
    assert controls.shape[0] == d + 1
    return powers_t(d - 1, t) @ BMdt[d] @ controls

In [ ]:
def megaflow(d: int, points: NDArray, tt: NDArray):
    wm = powers_tt(d - 1, tt) @ BMdt[d]
    return np.concat([wm @ controls_i(d, points, i) for i in range(d, len(points))])

In [ ]:
arrows_tt = np.arange(0.0, 1.0, 0.25)
tang_points = megacurve(3, points, arrows_tt)
tang_vectors = megaflow(3, points, arrows_tt)
tang_lengths = np.linalg.norm(tang_vectors, axis=1)

In [ ]:
tang_mesh = pv.PolyData(tang_points)
tang_mesh["vectors"] = tang_vectors
tang_mesh["lengths"] = tang_lengths
pvm_arrows = tang_mesh.glyph(orient="vectors", scale="lengths", factor=0.25)

In [ ]:
pl = pv.Plotter()
pl.background_color = pv.Color("#303030")
pl.show_grid()
_ = pl.add_mesh(pvm_controls, color="gray", line_width=2)
_ = pl.add_mesh(pvm_arrows, scalars="lengths")
pl.view_xy()
pl.show()